# LoRA 微调 Qwen2.5-1.5B-Instruct for RAG 问答

**Colab Free T4 GPU | 预计 1 小时**

### LoRA 参数
| 参数 | 值 | 原因 |
|------|-----|------|
| rank(r) | 8 | 论文默认,效果接近r=16 |
| alpha | 16 | alpha/r=2 标准缩放 |
| target | q_proj, v_proj | 注意力Q/V最关键 |
| lr | 2e-4 | 小模型常用学习率 |
| epochs | 3 | 100条数据,3轮防过拟合 |

## Step 1: 检查 GPU

In [ ]:
!pip install -q transformers peft accelerate bitsandbytes datasets
import torch
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem/1024**3:.1f} GB")

## Step 2: 加载训练数据

> 先上传 `training_data.json` (左侧文件面板 -> 上传按钮)

In [ ]:
import json, os

if not os.path.exists("training_data.json"):
    raise FileNotFoundError("请先上传 training_data.json!")

with open("training_data.json", "r", encoding="utf-8") as f:
    raw = json.load(f)

print(f"Samples: {len(raw['samples'])}")
print(f"Task: {raw['description']}")
print("OK")

## Step 3: 格式化为 ChatML

In [ ]:
SYSTEM_PROMPT = (
    "你是一个基于文档知识的问答助手。请严格依据提供的参考资料回答问题。\n"
    "规则：\n"
    "1. 只使用参考资料中的信息回答，不要编造\n"
    "2. 如果资料不足，请明确说'根据现有资料无法回答'\n"
    "3. 回答末尾标注来源编号，如 [来源:1,2]\n"
    "4. 回答简洁准确，避免冗余"
)

def format_samples(raw_data):
    formatted = []
    for s in raw_data["samples"]:
        text = (
            f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
            f"<|im_start|>user\n{s['instruction']}<|im_end|>\n"
            f"<|im_start|>assistant\n{s['output']}<|im_end|>"
        )
        formatted.append({"text": text})
    return formatted

formatted = format_samples(raw)
print(f"Formatted: {len(formatted)} samples")
print("Sample (first 200 chars):")
print(formatted[0]["text"][:200])

## Step 4: Tokenizer & 分词

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME, trust_remote_code=True, padding_side="right")
tokenizer.pad_token = tokenizer.eos_token

ds = Dataset.from_list(formatted)

def tokenize_fn(examples):
    return tokenizer(examples["text"], truncation=True,
                     max_length=1024, padding="max_length")

tokenized = ds.map(tokenize_fn, batched=True,
                   remove_columns=ds.column_names)
print(f"Tokenized: {len(tokenized)} samples, seq_len=1024")

## Step 5: 加载模型 (4-bit NF4)

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training
import torch

bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True)

print(f"Downloading {MODEL_NAME}... (~3GB)")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb,
    device_map="auto", trust_remote_code=True,
    torch_dtype=torch.float16)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)
print(f"Loaded. VRAM: {torch.cuda.memory_allocated()/1024**3:.1f} GB")

## Step 6: 配置 LoRA

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

config = LoraConfig(
    r=8, lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05, bias="none",
    task_type=TaskType.CAUSAL_LM)

model = get_peft_model(model, config)
model.print_trainable_parameters()
# Expected: ~3M trainable params out of ~1.5B total

## Step 7: 训练 (30-45 分钟)

In [ ]:
from transformers import TrainingArguments, Trainer

args = TrainingArguments(
    output_dir="./qwen25-rag-lora", num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4, warmup_steps=100,
    logging_steps=10, save_steps=100,
    fp16=True, report_to="none",
    save_total_limit=2)

trainer = Trainer(model=model, args=args,
                  train_dataset=tokenized, tokenizer=tokenizer)

print("Training...")
trainer.train()

# Save LoRA adapter
model.save_pretrained("./qwen25-rag-lora")
tokenizer.save_pretrained("./qwen25-rag-lora")
print("Saved!")

## Step 8: 微调后推理测试

In [ ]:
def generate(instruction, max_tokens=256):
    prompt = (
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\n{instruction}<|im_end|>\n"
        f"<|im_start|>assistant\n")
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs, max_new_tokens=max_tokens,
        temperature=0.3, do_sample=True, top_p=0.9,
        pad_token_id=tokenizer.eos_token_id)
    full = tokenizer.decode(outputs[0], skip_special_tokens=True)
    if "<|im_start|>assistant" in full:
        return full.split("<|im_start|>assistant")[-1].strip()
    return full

tests = [
    ("根据参考资料：[1] 机器学习是人工智能一个分支，使计算机从数据中学习。问题：什么是机器学习？", "概念解释"),
    ("根据参考资料：[1] RAG工作流程：文档处理->切片->向量化->检索->生成答案。问题：RAG工作流程是什么？", "事实查询"),
    ("根据参考资料：未找到相关内容。问题：明年的天气会怎样？", "拒绝回答"),
    ("根据参考资料：[1] 监督学习用标注数据。[2] 强化学习靠奖励机制。[3] 无监督学习发现隐藏结构。问题：机器学习有哪些类型？分别有什么特点？", "多信息综合"),
]

for q, label in tests:
    print(f"\n=== {label} ===")
    ans = generate(f"根据以下参考资料回答问题：\n\n{q}")
    print(f"A: {ans[:300]}")

## Step 9: 下载结果

In [ ]:
# Merge LoRA into base model
merged = model.merge_and_unload()
merged.save_pretrained("./qwen25-rag-merged", safe_serialization=True)
tokenizer.save_pretrained("./qwen25-rag-merged")

# Zip adapter for download
!zip -r qwen25-rag-lora.zip ./qwen25-rag-lora/
!ls -lh qwen25-rag-lora.zip
print("\nRight-click qwen25-rag-lora.zip -> Download")

## Step 10: 效果分析

### 对比

| 测试 | DeepSeek v4-flash | Qwen2.5原版 | Qwen2.5+LoRA |
|------|:--:|:--:|:--:|
| 引用来源 | ✅ | ❌ | ✅ |
| 拒绝编造 | ✅ | ❌ | ✅ |
| 结构化输出 | ✅ | ⚠️ | ✅ |

### 面试要点
- **微调提升什么？** 回答格式/风格/引用习惯，不是知识本身
- **LoRA局限？** 不注入新知识，只调整行为模式
- **微调 vs RAG？** 微调优化"怎么答"，RAG提供"答什么"